# §29 — parallel_cubic: cubic'i sıralı z-taraması olmadan yeniden formüle etmek

**Fikir.** Geri beslemeyi kapalı-döngüden açık-döngüye çevir:

    s_t   = λ₀·s_{t-1} + k_t                    (AFFINE, sabit katsayı → chunkwise paralel)
    λ_t   = λ₀ · (1 + 2η·s_{t-1}²)^(-1/2)

λ₀ = σ(decay) = exp'in öğrenilen kanal-başı decay'i → **yeni hiperparametre yok**;
s, "saf exp altında z ne olurdu" tahminidir.

**İç-içe (nested) model:** η=0 → λ_t = λ₀ = **tam olarak exp**. Yani ifade gücü
exp'ten düşük olamaz; "η sıfırdan uzaklaşmayı öğreniyor mu?" doğrudan teşhis.

**Kazanç:** sıralı özel op yok → exp/GLA ile birebir aynı paralellik yapısı →
mobil ihraç engeli kalkar (RESULTS §28b'nin 1. ve 2. raf-kaldırma koşulu).

**Risk (dürüst):** §28a'daki kazanç *uyarlanabilirlikten* mi geliyordu, yoksa tam
kapalı-döngü dinamiğinden mi? Bilmiyoruz — bu deney onu ölçüyor.

---
### ÖN-KAYITLI KRİTERLER (koşudan önce)
Protokol §28 ile **birebir aynı**: carry_curriculum, n=16 eşleşmiş seed, birincil
endpoint = eğitim-sonu cross-chunk doğrulama kaybı. Referans: §28a'da
cubic(sıralı) − exp = **+0.484 nat**.

- **RAF KALKAR:** parallel_cubic, exp'e karşı bu farkın **≥%70'ini** korursa
  (≥ +0.339 nat) VE p < 0.05 → hem kanıtlı hem sevk edilebilir; varsayılan
  seyrek-rejim mekanizması olur.
- **KISMİ:** %30–70 arası korunuyorsa → uyarlanabilirlik faydanın bir kısmını
  taşıyor ama kapalı-döngü de katkılı; ikisi de belgelenir, karar ertelenir.
- **BAŞARISIZ:** <%30 veya p ≥ 0.05 → kazanç kapalı-döngü geri beslemeye özgüymüş;
  parallel_cubic reddedilir, cubic rafta kalır. Dürüstçe böyle yazılır.

Ayrıca **Hücre 2 bir DOĞRULAMA KAPISI**: η=0'da parallel_cubic ≡ exp olmalı.
Geçmezse implementasyon hatalıdır ve deney koşulmaz.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, math
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
ROOT = os.path.join(BASE,'eta_sweep'); os.makedirs(ROOT, exist_ok=True)   # §28 ile ayni -> exp/cubic cache'i kullanilir
print('repo:', REPO, '| cikti:', ROOT)

In [ ]:
# --- 2. DOGRULAMA KAPISI (deneyden ONCE implementasyonu sina) ---
import torch, time
from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

def build(mode, seed=0):
    torch.manual_seed(seed)
    cfg = HFPConfig(vocab_size=164, hidden_size=64, num_hidden_layers=2,
                    num_attention_heads=2, intermediate_size=256, bulk_dim=32,
                    short_len=8, max_position_embeddings=264, local_window=8,
                    decay_mode=mode, rec_block=32, write_rule="additive",
                    key_feature_map="dpfp", pe_period=256)
    return HFPForCausalLM(cfg).eval()

from hfp.core.hfp_bulk_state import HFPBulkState
def bulks(m):                      # isimden bagimsiz: modul agacini tara
    return [mm for mm in m.modules() if isinstance(mm, HFPBulkState)]

x = torch.randint(1, 100, (2, 256))
me = build("exp", 0)
mp = build("parallel_cubic", 0); mp.load_state_dict(me.state_dict(), strict=False)

# T1: eta -> 0  =>  parallel_cubic TAM OLARAK exp olmali (ic-ice model iddiasi)
with torch.no_grad():
    for bs in bulks(mp): bs.log_eta.fill_(-30.0)                 # eta ~ 1e-13
    a = me(x).logits; b = mp(x).logits
d1 = (a-b).abs().max().item()
print(f'T1 (eta=0 -> exp ile ozdes): max|fark| = {d1:.3e}  -> {"GECTI" if d1 < 1e-4 else "KALDI"}')

# T2: eta > 0  =>  exp'ten FARKLI olmali (mekanizma gercekten devrede)
mp2 = build("parallel_cubic", 0); mp2.load_state_dict(me.state_dict(), strict=False)
with torch.no_grad(): c = mp2(x).logits
d2 = (a-c).abs().max().item()
print(f'T2 (eta>0 -> exp\'ten farkli): max|fark| = {d2:.3e} -> {"GECTI" if d2 > 1e-3 else "KALDI"}')

# T3: gradyan saglikli, NaN yok, log_eta gradyan aliyor
mp3 = build("parallel_cubic", 1); mp3.train()
out = mp3(x, labels=x); out.loss.backward()
ge = [p.grad for n,p in mp3.named_parameters() if 'log_eta' in n and p.grad is not None]
t3 = bool(torch.isfinite(out.loss)) and all(torch.isfinite(g).all() for g in ge) and any(g.abs().sum()>0 for g in ge)
print(f'T3 (loss finite + log_eta gradyani akiyor): {"GECTI" if t3 else "KALDI"}  loss={out.loss.item():.3f}')

# T4: HIZ — sirali cubic vs parallel_cubic vs exp
xl = torch.randint(1,100,(4,512))
sp = {}
for mode in ("cubic_flux_chunked","parallel_cubic","exp"):
    mm = build(mode,0)
    with torch.no_grad():
        mm(xl); t0=time.time()
        for _ in range(3): mm(xl)
        sp[mode]=(time.time()-t0)/3
    print(f'   {mode:>20}: {sp[mode]*1000:7.1f} ms/forward (B=4,L=512)')
print(f'   -> parallel_cubic, sirali cubic\'e gore {sp["cubic_flux_chunked"]/sp["parallel_cubic"]:.2f}x hizli; '
      f'exp\'e gore {sp["parallel_cubic"]/sp["exp"]:.2f}x yavas')

GATE = (d1 < 1e-4) and (d2 > 1e-3) and t3
print('\n' + ('KAPI GECILDI -> deney kosulabilir.' if GATE else
      'KAPI KALDI -> IMPLEMENTASYON HATALI. Deneyi KOSMA, once duzelt.'))
assert GATE, 'Dogrulama kapisi basarisiz'

In [ ]:
# --- 3. DENEY: n=16 eslesmis (exp / cubic sirali / parallel_cubic) ---
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'60'}
RE_VER = re.compile(r'cross-chunk dogrulama loss:\s*([0-9.]+)')
N_SEEDS = 16
ARMS = [('exp','exp_reference'), ('cubic_flux_chunked','cubic_eta_default'),
        ('parallel_cubic','parallel_cubic')]
data = {}
for mode, tag in ARMS:
    ck = os.path.join(ROOT, tag); os.makedirs(ck, exist_ok=True)
    env = {**BASE_ENV, 'HFP_CKPT_DIR': ck}
    env.pop('HFP_ETA_LOG_MIN',None); env.pop('HFP_ETA_LOG_MAX',None)
    data[tag] = {}
    for s in range(N_SEEDS):
        cache = os.path.join(ck, f'result_s{s}.json')
        if os.path.exists(cache):
            data[tag][s] = json.load(open(cache)); continue
        print(f'[{tag} s{s}] ...', end=' ', flush=True)
        r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',mode,str(s),'6000'],
                           cwd=REPO, env=env, capture_output=True, text=True)
        mv = RE_VER.search(r.stdout + r.stderr)
        d = {'loss': float(mv.group(1)), 'nan': False} if mv else {'loss': None, 'nan': True}
        print(f'{d["loss"]:.3f}' if mv else 'IRAKSADI', flush=True)
        json.dump(d, open(cache,'w')); data[tag][s] = d
json.dump(data, open(os.path.join(ROOT,'parallel_cubic_n16.json'),'w'), indent=2)
print('\nDENEY TAMAM')

In [ ]:
# --- 4. ON-KAYITLI HUKUM (§29) ---
import statistics as st, math
from math import lgamma
def betacf(a,b,x):
    MAXIT,EPS,FPMIN=200,3e-12,1e-300
    qab,qap,qam=a+b,a+1,a-1; c=1.0; dd=1-qab*x/qap
    if abs(dd)<FPMIN: dd=FPMIN
    dd=1/dd; h=dd
    for mm in range(1,MAXIT+1):
        m2=2*mm
        aa=mm*(b-mm)*x/((qam+m2)*(a+m2)); dd=1+aa*dd; c=1+aa/c
        if abs(dd)<FPMIN: dd=FPMIN
        if abs(c)<FPMIN: c=FPMIN
        dd=1/dd; h*=dd*c
        aa=-(a+mm)*(qab+mm)*x/((a+m2)*(qap+m2)); dd=1+aa*dd; c=1+aa/c
        if abs(dd)<FPMIN: dd=FPMIN
        if abs(c)<FPMIN: c=FPMIN
        dd=1/dd; de=dd*c; h*=de
        if abs(de-1)<EPS: break
    return h
def tp(t,df):
    x=df/(df+t*t); a,b=df/2,0.5
    if x<(a+1)/(a+b+2):
        return math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+a*math.log(x)+b*math.log(1-x))*betacf(a,b,x)/a
    return 1-math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+b*math.log(1-x)+a*math.log(x))*betacf(b,a,1-x)/b

def paired(tag_a, tag_b):     # (a - b): pozitif = b daha iyi (kayip dusuk)
    A,B = data[tag_a], data[tag_b]
    pr = [(A[s]['loss'], B[s]['loss']) for s in range(N_SEEDS)
          if not A[s]['nan'] and not B[s]['nan']]
    d = [a-b for a,b in pr]; n=len(d)
    t = st.mean(d)/(st.stdev(d)/math.sqrt(n))
    return st.mean(d), sum(1 for x in d if x>0), n, t, tp(abs(t), n-1)

for tag in ARMS:
    L=[data[tag[1]][s]['loss'] for s in range(N_SEEDS) if not data[tag[1]][s]['nan']]
    nan=sum(1 for s in range(N_SEEDS) if data[tag[1]][s]['nan'])
    print(f'{tag[1]:>20}: loss {st.mean(L):.3f}  (n={len(L)}, iraksama={nan})')

REF = 0.484        # §28a: cubic(sirali) - exp
print('\n=== ON-KAYITLI HUKUM (§29) ===')
mp_, w_, n_, t_, p_ = paired('exp_reference','parallel_cubic')
mc_, wc, nc, tc, pc = paired('exp_reference','cubic_eta_default')
print(f'parallel_cubic vs exp : {mp_:+.3f} nat, {w_}/{n_} seed, t={t_:.2f}, p={p_:.4f}')
print(f'cubic(sirali)  vs exp : {mc_:+.3f} nat, {wc}/{nc} seed, t={tc:.2f}, p={pc:.4f}  (§28a tekrari)')
keep = mp_/REF*100
print(f'\nKorunan fayda: {keep:.0f}% (referans +{REF} nat)')
if keep>=70 and p_<0.05:
    print('RAF KALKAR: parallel_cubic faydanin >=%70\'ini koruyor ve anlamli.')
    print('  -> Hem kanitli hem sevk edilebilir; seyrek-rejim varsayilan mekanizmasi olur.')
    print('  -> Sonraki: mobil ihrac yolu (GGUF/ExecuTorch) + LM olceginde tekrar.')
elif keep>=30:
    print('KISMI: uyarlanabilirlik faydanin bir kismini tasiyor, kapali-dongu de katkili.')
    print('  -> Ikisi de belgelenir; raf karari ertelenir.')
else:
    print('BASARISIZ: kazanc kapali-dongu geri beslemeye ozguymus.')
    print('  -> parallel_cubic reddedilir, cubic rafta kalir. Durustce boyle yazilir.')